<a href="https://colab.research.google.com/github/AndresAvella09/Proyecto_Multimodal/blob/main/notebooks/06_diagnostico_modal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 — Diagnóstico modal con las etiquetas actuales (Tier B, bloque 1)

El Exp. 1 (notebook 04) mostró que la fusión ingenua **sí supera** al texto (+4.5 ± 0.2 puntos de F1
macro): no hay *greedy modality* en su forma clásica. Ganarle al texto no implica usar bien la
imagen, así que este notebook mide **cuánto usa la fusión cada modalidad** y **cuánta información
aporta la imagen**:

| Sección | Pregunta | Herramienta | Lectura del brief |
|---|---|---|---|
| A | ¿De qué rama depende la fusión? | Ablación: apagar una rama en test | Diagnóstico |
| B | ¿Concatenar extrae más que dos expertos separados? | Fusión tardía (promedio y pesos aprendidos en dev) | Base del método |
| C | ¿Cuánta información sobre el género aporta la imagen si ya tengo el texto? | Estimación de I(X_img; Y \| X_txt) con log-loss | Lectura A |
| D | ¿El texto está sobreconfiado y la imagen subconfiada? | ECE, diagramas de confiabilidad, temperatura | Lectura C |
| E | ¿Cambia la dependencia de cada modalidad con la época de la película? | A, C y D por época sobre el test | Nivel 1 (notebook 05) |

**Datos actuales (notebook 05):** por defecto se usan las **etiquetas de IMDb hoy** (`hoy_*`, máximo
3 géneros) de `mmimdb_con_anio.parquet`, y las métricas se calculan sobre los **géneros de
`generos_comparacion.json`** (los que tienen suficientes películas recientes). Así las cifras de
este notebook son comparables con las de las películas 2020–2026 y la muestra control de TMDB.
Con `ETIQUETAS = 'originales'` se reproduce el protocolo del notebook 04.

Los modelos se entrenan con los 23 géneros; el recorte a los géneros de comparación afecta solo a
la evaluación. Se guardan pesos, estandarización, umbrales y temperaturas para aplicar los modelos
a las películas nuevas **sin reentrenar**.

El código común (datos, modelo, entrenamiento, métricas) vive en `src/` del repo. Corre en CPU (~5 min).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time, subprocess
REPO = '/content/Proyecto_Multimodal'
if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/AndresAvella09/Proyecto_Multimodal.git', REPO], check=True)
sys.path.insert(0, REPO)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from src.datos import cargar, estandarizar
from src.modelos import entrenar, predecir_logits
from src.metricas import (sigmoide, f1_por_clase, elegir_umbrales, metricas, ap_por_clase, ic95,
                          bootstrap_media, bce_por_etiqueta, ece, ece_macro, curva_confiabilidad,
                          ajustar_temperatura)

DATA = '/content/drive/MyDrive/proyecto_multimodal/data'
EMB = '/content/drive/MyDrive/proyecto_multimodal/embeddings'
RES = '/content/drive/MyDrive/proyecto_multimodal/results'
os.makedirs(RES, exist_ok=True)

ETIQUETAS = 'hoy'                # 'hoy' (IMDb actual, notebook 05) u 'originales' (MM-IMDB, notebook 04)
SOLO_GENEROS_COMPARACION = True  # evaluar solo en los géneros de generos_comparacion.json

X, Y, idx, GENEROS, meta = cargar(DATA, EMB, etiquetas=ETIQUETAS)
Xz, stats_estandarizacion = estandarizar(X, idx['train'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'

if SOLO_GENEROS_COMPARACION:
    with open(f'{DATA}/generos_comparacion.json') as fh:
        comparacion = json.load(fh)
    GENEROS_EVAL = comparacion['generos_comparacion']
    print("Fuera de la evaluación:", comparacion['excluidos'])
else:
    GENEROS_EVAL = GENEROS
GE = np.array([GENEROS.index(g) for g in GENEROS_EVAL])   # columnas que se evalúan

print(f"Etiquetas: {ETIQUETAS} | {len(GENEROS)} géneros para entrenar, {len(GENEROS_EVAL)} para evaluar")
print({s: len(i) for s, i in idx.items()}, "| device:", device)
print(f"Géneros por película en train: {Y[idx['train']].sum(1).mean():.2f}")

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3, 'figure.dpi': 110})
COLORES = {'imagen': '#f59e0b', 'texto': '#2563eb', 'fusion_ingenua': '#10b981',
           'tardia_promedio': '#8b5cf6', 'tardia_global': '#ec4899', 'tardia_por_genero': '#64748b'}

## 0. Entrenamiento (mismo protocolo y configuración que el 04)

Se guardan los **logits** de dev y test de cada corrida, ya recortados a los géneros de evaluación:
todo lo que sigue se calcula a partir de ellos sin volver a entrenar. Con `ETIQUETAS = 'originales'`
y los 23 géneros, al final se comparan los F1 con los del 04 como chequeo de que `src/` reproduce
el mismo experimento.

In [ ]:
CONFIG = {'dim_proy': 512, 'dropout': 0.3, 'lr': 1e-3, 'weight_decay': 1e-4, 'batch': 256,
          'max_epocas': 100, 'paciencia': 10, 'semillas': [0, 1, 2, 3, 4],
          'umbrales': np.round(np.arange(0.05, 0.96, 0.05), 2), 'n_bootstrap': 1000}
MODELOS = {'imagen': ['img'], 'texto': ['txt'], 'fusion_ingenua': ['img', 'txt']}
SEMILLAS = CONFIG['semillas']

def tensores(mods, s, apagar=()):
    # apagar: modalidades reemplazadas por su media de train (= vector cero tras estandarizar)
    return [torch.zeros(len(idx[s]), Xz[m].shape[1], device=device) if m in apagar
            else torch.from_numpy(Xz[m][idx[s]]).to(device) for m in mods]

Ytr = torch.from_numpy(Y[idx['train']]).to(device)       # se entrena con los 23 géneros
Ydev, Yte = Y[idx['dev']][:, GE], Y[idx['test']][:, GE]   # se evalúa en GENEROS_EVAL

def logits_eval(modelo, mods, p, apagar=()):
    return predecir_logits(modelo, tensores(mods, p, apagar))[:, GE]

modelos, logits = {}, {}   # claves (nombre, semilla); logits[(n, s)] = {'dev': ..., 'test': ...}
t0 = time.time()
for s in SEMILLAS:
    for n, mods in MODELOS.items():
        modelo, _ = entrenar(tensores(mods, 'train'), Ytr, tensores(mods, 'dev'), Y[idx['dev']], CONFIG, s)
        modelos[(n, s)] = modelo
        logits[(n, s)] = {p: logits_eval(modelo, mods, p) for p in ['dev', 'test']}
print(f"Entrenamiento: {time.time() - t0:.0f} s")

umbrales = {}
def evaluar(lg_dev, lg_test, clave=None):
    # Umbrales elegidos en dev, métricas en test
    umb = elegir_umbrales(Ydev, sigmoide(lg_dev), CONFIG['umbrales'])
    if clave is not None:
        umbrales[clave] = umb
    return metricas(Yte, sigmoide(lg_test), umb)

base = pd.DataFrame([{'modelo': n, 'semilla': s, **evaluar(logits[(n, s)]['dev'], logits[(n, s)]['test'], (n, s))}
                     for s in SEMILLAS for n in MODELOS])
resumen = base.groupby('modelo')[['f1_macro', 'ap_macro']].mean().loc[list(MODELOS)]

ruta_exp1 = f'{RES}/exp1_baselines_por_semilla.csv'
if ETIQUETAS == 'originales' and len(GENEROS_EVAL) == len(GENEROS) and os.path.exists(ruta_exp1):
    resumen['f1_macro_exp1'] = pd.read_csv(ruta_exp1).groupby('modelo')['test_f1_macro'].mean()
(100 * resumen).round(2)

## A. ¿De qué rama depende la fusión? (ablación)

En el modelo de fusión ya entrenado se **apaga** una rama en test: su embedding se reemplaza por la
media de train (el vector cero, porque los datos están estandarizados). La caída de rendimiento mide
cuánto se apoya la fusión en esa rama.

- Se reporta la caída de **AP macro** (no depende de umbrales). Los umbrales de dev se eligieron
  con las dos ramas encendidas, así que el F1 de un modelo con una rama apagada mezclaría "perder
  la modalidad" con "umbrales desajustados".
- **Índice de dependencia de la imagen** = caída_img / (caída_img + caída_txt). 0.5 = reparto
  parejo; cerca de 0 = la fusión casi ignora la imagen.
- Comparación clave: **fusión sin imagen vs modelo solo texto**. Si la fusión sin imagen rinde
  *peor* que el modelo de texto entrenado solo, la rama de texto dentro de la fusión aprendió menos
  que por su cuenta, porque se apoya en la imagen para parte del trabajo.

Limitación: reemplazar por la media es una intervención fuera de la distribución de entrenamiento
(el modelo nunca vio una rama en cero). Mide dependencia, no un contrafactual exacto.

In [ ]:
filas_abl = []
for s in SEMILLAS:
    m = modelos[('fusion_ingenua', s)]
    ap = {'completa': ap_por_clase(Yte, logits[('fusion_ingenua', s)]['test'])}
    for nombre, apagar in [('sin_imagen', ['img']), ('sin_texto', ['txt'])]:
        ap[nombre] = ap_por_clase(Yte, logits_eval(m, ['img', 'txt'], 'test', apagar))
    ap['solo_texto'] = ap_por_clase(Yte, logits[('texto', s)]['test'])
    ap['solo_imagen'] = ap_por_clase(Yte, logits[('imagen', s)]['test'])
    filas_abl.append({'semilla': s, **{k: v for k, v in ap.items()}})

def media_sem(k):
    return np.stack([f[k] for f in filas_abl])          # (semillas, C)

caida_img = media_sem('completa') - media_sem('sin_imagen')    # (semillas, C)
caida_txt = media_sem('completa') - media_sem('sin_texto')
indice = caida_img.mean(1) / (caida_img.mean(1) + caida_txt.mean(1))

print("AP macro en test (media ± IC 95% sobre semillas):")
for k in ['completa', 'sin_imagen', 'sin_texto', 'solo_texto', 'solo_imagen']:
    m, h = ic95(media_sem(k).mean(1)); print(f"  {k:12s} {100 * m:6.2f} ± {100 * h:.2f}")
for nombre, c in [('imagen', caida_img), ('texto', caida_txt)]:
    m, h = ic95(c.mean(1)); print(f"Caída al apagar {nombre:7s}: {100 * m:6.2f} ± {100 * h:.2f} puntos de AP macro")
m, h = ic95(indice); print(f"\nÍndice de dependencia de la imagen: {m:.3f} ± {h:.3f}")
m, h = ic95((media_sem('sin_imagen') - media_sem('solo_texto')).mean(1))
print(f"Fusión sin imagen − solo texto: {100 * m:+.2f} ± {100 * h:.2f} puntos de AP macro")

In [ ]:
abl_genero = pd.DataFrame({'caida_al_apagar_imagen': caida_img.mean(0),
                           'caida_al_apagar_texto': caida_txt.mean(0),
                           'ventaja_imagen_unimodal': (media_sem('solo_imagen') - media_sem('solo_texto')).mean(0)},
                          index=GENEROS_EVAL).sort_values('caida_al_apagar_imagen')

fig, ax = plt.subplots(figsize=(9, 7.5))
y = np.arange(len(abl_genero))
ax.barh(y - 0.2, 100 * abl_genero['caida_al_apagar_imagen'], height=0.4, color=COLORES['imagen'], label='apagar imagen')
ax.barh(y + 0.2, 100 * abl_genero['caida_al_apagar_texto'], height=0.4, color=COLORES['texto'], label='apagar texto')
ax.set_yticks(y, abl_genero.index, fontsize=8); ax.grid(axis='y', visible=False)
ax.set_xlabel('caída de AP en test (puntos, media de 5 semillas)')
ax.set_title('¿Cuánto pierde la fusión al apagar cada rama?'); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

# ¿La dependencia de la imagen sigue a su utilidad real? Correlación entre géneros
r = np.corrcoef(abl_genero['ventaja_imagen_unimodal'], abl_genero['caida_al_apagar_imagen'])[0, 1]
print(f"Correlación (entre géneros) ventaja unimodal de la imagen vs dependencia de la fusión: r = {r:.2f}")

## B. Fusión tardía: dos expertos separados

Se combinan los **logits** de las cabezas de imagen y texto ya entrenadas (misma semilla):

`logit = w · logit_img + (1 − w) · logit_txt`

| Variante | Cómo se elige w |
|---|---|
| `tardia_promedio` | w = 0.5 fijo |
| `tardia_global` | un w para todos los géneros, el que maximiza AP macro en **dev** |
| `tardia_por_genero` | un w por género, el que maximiza el AP de ese género en **dev** |

Se combinan logits y no probabilidades porque sumar logits equivale a multiplicar las razones de
odds de cada experto (un *producto de expertos*). Los pesos por género son una versión primitiva del
método propuesto: ahí w saldrá de la incertidumbre de cada rama en vez de buscarse en una grilla.

In [ ]:
GRILLA_W = np.round(np.arange(0, 1.0001, 0.05), 2)
filas_tardia, pesos_genero = [], []
for s in SEMILLAS:
    li, lt = logits[('imagen', s)], logits[('texto', s)]
    mezcla = lambda w, p: w * li[p] + (1 - w) * lt[p]

    ap_dev = np.stack([ap_por_clase(Ydev, mezcla(w, 'dev')) for w in GRILLA_W])   # (W, C)
    w_global = GRILLA_W[ap_dev.mean(1).argmax()]
    w_genero = GRILLA_W[ap_dev.argmax(0)]                                         # (C,)
    pesos_genero.append(w_genero)

    for nombre, w in [('tardia_promedio', 0.5), ('tardia_global', w_global), ('tardia_por_genero', w_genero)]:
        filas_tardia.append({'modelo': nombre, 'semilla': s, 'w_img': float(np.mean(w)),
                             **evaluar(mezcla(w, 'dev'), mezcla(w, 'test'))})

comp = pd.concat([base, pd.DataFrame(filas_tardia)], ignore_index=True)
tabla_b = []
for n in list(MODELOS) + ['tardia_promedio', 'tardia_global', 'tardia_por_genero']:
    r = comp[comp['modelo'] == n]; fila = {'modelo': n}
    for k in ['f1_macro', 'f1_micro', 'ap_macro']:
        m, h = ic95(r[k]); fila[k] = f"{100 * m:.2f} ± {100 * h:.2f}"
    fila['w_img medio'] = f"{r['w_img'].mean():.2f}" if 'w_img' in r and r['w_img'].notna().any() else ''
    tabla_b.append(fila)
pd.DataFrame(tabla_b).set_index('modelo')

In [ ]:
# Diferencias pareadas por semilla contra la fusión ingenua
fi = comp[comp['modelo'] == 'fusion_ingenua'].set_index('semilla')
for n in ['tardia_promedio', 'tardia_global', 'tardia_por_genero']:
    t = comp[comp['modelo'] == n].set_index('semilla')
    m1, h1 = ic95(t['f1_macro'] - fi['f1_macro']); m2, h2 = ic95(t['ap_macro'] - fi['ap_macro'])
    print(f"{n:18s} − fusión ingenua:  F1 macro {100 * m1:+.2f} ± {100 * h1:.2f} | AP macro {100 * m2:+.2f} ± {100 * h2:.2f}")

# Peso de la imagen elegido en dev por género vs dependencia medida con la ablación
w_gen = pd.Series(np.mean(pesos_genero, axis=0), index=GENEROS_EVAL)
print(f"\nCorrelación w_img por género (tardía) vs caída al apagar imagen (fusión): "
      f"r = {np.corrcoef(w_gen[abl_genero.index], abl_genero['caida_al_apagar_imagen'])[0, 1]:.2f}")
print("\nGéneros con mayor peso de imagen (tardía por género):")
print(w_gen.sort_values(ascending=False).head(6).round(2).to_string())

## C. ¿Cuánta información aporta la imagen? Estimación de I(X_img; Y | X_txt)

La información mutua condicional se puede escribir como diferencia de entropías condicionales:

I(X_img; Y | X_txt) = H(Y | X_txt) − H(Y | X_img, X_txt)

La **log-loss en test** de un modelo q es la entropía cruzada E[−log q(Y|X)], que es una **cota
superior** de H(Y|X) (la diferencia es la divergencia KL entre la verdad y el modelo). Entonces:

Î = log-loss(texto) − log-loss(fusión)

**Ojo con la interpretación:** cada término es una cota superior, pero **su diferencia no es una
cota** de nada: es una estimación plug-in que es buena si los dos modelos se acercan igual de bien a
sus condicionales verdaderas. Se reporta como tal, con IC bootstrap sobre películas de test.

**Calibración antes de restar.** Un modelo sobreconfiado infla su log-loss (el término KL), y como
los modelos se eligen por AP y no por log-loss, la fusión, que tiene más parámetros, puede quedar
más sobreconfiada que el texto. Eso sesga Î hacia abajo y puede volverla negativa. Por eso la
estimación principal usa logits **escalados por temperatura** (T ajustada en dev para cada modelo y
semilla, sección D): achica el término KL de cada modelo sin tocar el test. Se reporta también la
versión sin calibrar.

La misma cuenta da una descomposición completa (en nats por película, sumando los géneros de
evaluación, que se tratan como condicionalmente independientes dado X):

```
log-loss(prior)  →  log-loss(texto)  →  log-loss(fusión)
   H(Y)             H(Y | X_txt)        H(Y | X_img, X_txt)
        \_ Î(X_txt; Y) _/   \_ Î(X_img; Y | X_txt) _/
```

donde *prior* predice siempre la prevalencia de train de cada género (no mira la película).

In [ ]:
prev = Y[idx['train']].mean(0)[GE]
logit_prior = np.tile(np.log(prev / (1 - prev)), (len(Yte), 1))

TEMP = {(n, s): ajustar_temperatura(Ydev, logits[(n, s)]['dev']) for s in SEMILLAS for n in MODELOS}

def perdida_test(n, calibrar):
    # (n_test, C): BCE por película y género, promediada sobre semillas
    return np.mean([bce_por_etiqueta(Yte, logits[(n, s)]['test'] / (TEMP[(n, s)] if calibrar else 1.0))
                    for s in SEMILLAS], axis=0)

filas_cmi, estim_por_version = [], {}
for version, calibrar in [('calibrado', True), ('sin_calibrar', False)]:
    L = {'prior': bce_por_etiqueta(Yte, logit_prior),
         **{n: perdida_test(n, calibrar) for n in MODELOS}}
    estim = {
        'I(X_txt; Y)':          L['prior'] - L['texto'],
        'I(X_img; Y)':          L['prior'] - L['imagen'],
        'I(X_img; Y | X_txt)':  L['texto'] - L['fusion_ingenua'],
        'I(X_txt; Y | X_img)':  L['imagen'] - L['fusion_ingenua'],
    }
    estim_por_version[version] = estim
    print(f"=== {version} ===")
    print("Log-loss en test (nats por película, suma de", len(GENEROS_EVAL), "géneros):",
          {k: round(float(v.sum(1).mean()), 4) for k, v in L.items()})
    for k, v in estim.items():
        m, lo, hi = bootstrap_media(v.sum(1), CONFIG['n_bootstrap'])
        filas_cmi.append({'version': version, 'cantidad': k, 'nats': m, 'ic_inf': lo, 'ic_sup': hi, 'bits': m / np.log(2)})
        print(f"  {k:22s} {m:.4f}  [{lo:.4f}, {hi:.4f}]   = {m / np.log(2):.4f} bits")
    print()

estim = estim_por_version['calibrado']   # versión principal para lo que sigue
i_img, i_img_txt = estim['I(X_img; Y)'].sum(1).mean(), estim['I(X_img; Y | X_txt)'].sum(1).mean()
lo_img = [f for f in filas_cmi if f['version'] == 'calibrado' and f['cantidad'] == 'I(X_img; Y)'][0]['ic_inf']
if lo_img > 0:   # el cociente solo tiene sentido si la imagen sola aporta información detectable
    print(f"De la información que la imagen tiene sola, queda {i_img_txt / i_img:.0%} como aporte nuevo "
          "una vez que se tiene el texto (versión calibrada)")

In [ ]:
# Por género: aporte de la imagen dado el texto
m, lo, hi = bootstrap_media(estim['I(X_img; Y | X_txt)'], CONFIG['n_bootstrap'])
cmi_genero = pd.DataFrame({'nats': m, 'ic_inf': lo, 'ic_sup': hi}, index=GENEROS_EVAL).sort_values('nats')

fig, ax = plt.subplots(figsize=(8, 7))
y = np.arange(len(cmi_genero))
ax.barh(y, 1000 * cmi_genero['nats'], color=COLORES['imagen'], height=0.7)
ax.errorbar(1000 * cmi_genero['nats'], y, color='black', lw=1, fmt='none',
            xerr=[1000 * (cmi_genero['nats'] - cmi_genero['ic_inf']), 1000 * (cmi_genero['ic_sup'] - cmi_genero['nats'])])
ax.axvline(0, color='black', lw=0.8)
ax.set_yticks(y, cmi_genero.index, fontsize=8); ax.grid(axis='y', visible=False)
ax.set_xlabel('Î(X_img; Y_género | X_txt)  [milinats por película, IC 95% bootstrap]')
ax.set_title('Información que la imagen agrega sobre cada género, dado el texto')
plt.tight_layout(); plt.show()
print("Géneros con IC que incluye 0 (sin aporte detectable):",
      list(cmi_genero.index[(cmi_genero['ic_inf'] <= 0) & (cmi_genero['ic_sup'] >= 0)]))

## D. Calibración por modalidad

Premisa del método propuesto: *la modalidad fuerte es sobreconfiada y la débil subconfiada*. Se pone
a prueba con tres medidas, todas en test:

- **ECE** (todas las celdas película×género juntas) y **ECE macro** (por género y promediado, para
  que Drama no domine).
- **Diagramas de confiabilidad**: confianza declarada vs frecuencia observada. Debajo de la diagonal
  = sobreconfiado.
- **Temperatura óptima T** ajustada en **dev**: el factor que hay que dividir los logits para
  minimizar la log-loss. **T > 1 = sobreconfiado**, **T < 1 = subconfiado**. Es un resumen de un
  solo número con interpretación directa, y además se ve cuánto mejora el ECE en test al aplicarla.

Los modelos se seleccionaron por AP en dev (early stopping), no por log-loss: eso tiende a dejar
modelos algo sobreconfiados, y aplica igual a los tres.

In [ ]:
filas_cal = []
for s in SEMILLAS:
    for n in MODELOS:
        lg = logits[(n, s)]
        T = ajustar_temperatura(Ydev, lg['dev'])
        p, p_T = sigmoide(lg['test']), sigmoide(lg['test'] / T)
        filas_cal.append({'modelo': n, 'semilla': s, 'T': T,
                          'ece': ece(Yte, p), 'ece_macro': ece_macro(Yte, p),
                          'ece_tras_T': ece(Yte, p_T), 'ece_macro_tras_T': ece_macro(Yte, p_T),
                          'logloss': bce_por_etiqueta(Yte, lg['test']).sum(1).mean(),
                          'logloss_tras_T': bce_por_etiqueta(Yte, lg['test'] / T).sum(1).mean()})
cal = pd.DataFrame(filas_cal)

tabla_d = []
for n in MODELOS:
    r = cal[cal['modelo'] == n]; fila = {'modelo': n}
    for k in ['T', 'ece', 'ece_tras_T', 'ece_macro', 'ece_macro_tras_T', 'logloss', 'logloss_tras_T']:
        m, h = ic95(r[k]); esc = 1 if k.startswith(('T', 'logloss')) else 100
        fila[k] = f"{esc * m:.3f} ± {esc * h:.3f}" if esc == 1 else f"{esc * m:.2f} ± {esc * h:.2f}"
    tabla_d.append(fila)
print("ECE en puntos porcentuales; T y log-loss (nats por película) en su escala")
pd.DataFrame(tabla_d).set_index('modelo')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)
for ax, n in zip(axes, MODELOS):
    p = sigmoide(logits[(n, 0)]['test'])
    conf, acc, nb = curva_confiabilidad(Yte, p)
    ok = nb > 0
    ax.plot([0, 1], [0, 1], color='black', lw=0.8, ls='--')
    ax.plot(conf[ok], acc[ok], marker='o', color=COLORES[n])
    for c, a, k in zip(conf[ok], acc[ok], nb[ok]):
        ax.annotate(f"{k}", (c, a), fontsize=6, xytext=(3, -8), textcoords='offset points', color='gray')
    T = cal[(cal['modelo'] == n) & (cal['semilla'] == 0)]['T'].item()
    ax.set_title(f"{n}  (T = {T:.2f})"); ax.set_xlabel('confianza declarada')
axes[0].set_ylabel('frecuencia observada')
fig.suptitle('Diagramas de confiabilidad en test (semilla 0; número = predicciones en el bin)')
plt.tight_layout(); plt.show()

## E. ¿Cambia la dependencia con la época? (Nivel 1, dentro de MM-IMDB)

Se repiten A, C y la calibración **por época de estreno** sobre el test, con los mismos modelos
(no se reentrena) y las épocas del notebook 05. Es el primer paso de la pregunta temporal: si la
dependencia de la imagen o su aporte de información ya cambian *dentro* de MM-IMDB, hay que tenerlo
en cuenta antes de comparar con las películas 2020–2026.

Cuidados para que la comparación entre épocas sea justa:
- Se usan solo los géneros con al menos `MIN_POSITIVOS` películas **en todas las épocas**: con
  pocos positivos el AP de un género salta mucho por azar.
- La mezcla de géneros cambia con la época (notebook 05, sección 3), así que una diferencia en Î
  puede venir de qué géneros predominan y no de cómo se usa la imagen. Se reporta también el AP
  de cada modelo solo para ver si la tarea es más fácil o difícil en cada época.
- Los IC son sobre semillas: no incluyen la variabilidad por muestra de test, que pesa más en las
  épocas con menos películas (se informa `n`).

In [ ]:
MIN_POSITIVOS = 10
epoca = pd.cut(meta['anio'], bins=[0, 1979, 1999, 2009, 2100],
               labels=['antes de 1980', '1980-1999', '2000-2009', '2010 en adelante'])
epoca_test = np.asarray(epoca)[idx['test']]
EPOCAS = [e for e in epoca.cat.categories if (epoca_test == e).sum() > 0]

# Géneros con suficientes positivos en todas las épocas
pos = np.stack([Yte[epoca_test == e].sum(0) for e in EPOCAS])
ok = (pos >= MIN_POSITIVOS).all(0)
print(f"Géneros comparables entre épocas: {ok.sum()} de {len(GENEROS_EVAL)}")
print("Fuera:", [g for g, k in zip(GENEROS_EVAL, ok) if not k])

# Logits de la fusión con una rama apagada (una vez por semilla)
apagado = {(s, r): logits_eval(modelos[('fusion_ingenua', s)], ['img', 'txt'], 'test', [r])
           for s in SEMILLAS for r in ['img', 'txt']}

filas_ep = []
for e in EPOCAS:
    sel = epoca_test == e
    Ys = Yte[sel][:, ok]
    for s in SEMILLAS:
        lg = {n: logits[(n, s)]['test'][sel][:, ok] for n in MODELOS}
        ap = {n: ap_por_clase(Ys, lg[n]).mean() for n in MODELOS}
        caida_i = ap['fusion_ingenua'] - ap_por_clase(Ys, apagado[(s, 'img')][sel][:, ok]).mean()
        caida_t = ap['fusion_ingenua'] - ap_por_clase(Ys, apagado[(s, 'txt')][sel][:, ok]).mean()
        filas_ep.append({
            'epoca': e, 'semilla': s, 'n': int(sel.sum()),
            **{f'ap_{n}': ap[n] for n in MODELOS},
            'caida_al_apagar_imagen': caida_i, 'caida_al_apagar_texto': caida_t,
            'indice_dependencia_imagen': caida_i / (caida_i + caida_t),
            'cmi_img_dado_txt': (bce_por_etiqueta(Ys, lg['texto'] / TEMP[('texto', s)])
                                 - bce_por_etiqueta(Ys, lg['fusion_ingenua'] / TEMP[('fusion_ingenua', s)])).sum(1).mean(),
            **{f'ece_{n}': ece(Ys, sigmoide(lg[n])) for n in MODELOS},
        })
por_epoca = pd.DataFrame(filas_ep)

cols = ['ap_imagen', 'ap_texto', 'ap_fusion_ingenua', 'indice_dependencia_imagen', 'cmi_img_dado_txt',
        'ece_imagen', 'ece_texto', 'ece_fusion_ingenua']
tabla_e = por_epoca.groupby('epoca', sort=False).agg(
    n=('n', 'first'), **{c: (c, lambda x: '{:.3f} ± {:.3f}'.format(*ic95(x))) for c in cols})
tabla_e

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
x = np.arange(len(EPOCAS))
for ax, col, titulo in [(axes[0], 'indice_dependencia_imagen', 'Índice de dependencia de la imagen'),
                        (axes[1], 'cmi_img_dado_txt', 'Î(X_img; Y | X_txt)  [nats/película]')]:
    m_h = [ic95(por_epoca[por_epoca['epoca'] == e][col]) for e in EPOCAS]
    ax.errorbar(x, [m for m, _ in m_h], yerr=[h for _, h in m_h], marker='o', color=COLORES['imagen'], capsize=4)
    ax.set_xticks(x, EPOCAS, fontsize=8); ax.set_title(titulo)
for n in MODELOS:
    m_h = [ic95(por_epoca[por_epoca['epoca'] == e][f'ap_{n}']) for e in EPOCAS]
    axes[2].errorbar(x, [m for m, _ in m_h], yerr=[h for _, h in m_h], marker='o', color=COLORES[n], capsize=4, label=n)
axes[2].set_xticks(x, EPOCAS, fontsize=8); axes[2].set_title('AP macro en test'); axes[2].legend(fontsize=8)
fig.suptitle('Diagnóstico por época de estreno (test, media ± IC 95% sobre semillas)')
plt.tight_layout(); plt.show()

## Resumen y guardado

Tabla única con las cifras clave, y archivos para el paper. Todos llevan el sufijo de las etiquetas
usadas (`_hoy` u `_originales`) para que correr las dos versiones no pise resultados:

- `exp2_<etiquetas>_ablacion_por_genero.csv`, `..._fusion_tardia_por_semilla.csv`, `..._cmi.csv`,
  `..._cmi_por_genero.csv`, `..._calibracion_por_semilla.csv`, `..._por_epoca.csv`
- `exp2_<etiquetas>_logits.npz`: logits de dev y test de cada corrida (géneros de evaluación).
- `exp2_<etiquetas>_artefactos.pt`: **pesos de los 15 modelos, media y desviación de train, umbrales
  y temperaturas**. Es lo que se necesita para evaluar las películas de TMDB sin reentrenar:
  estandarizar con `aplicar_estandarizacion` → `predecir_logits` → recortar a los géneros de
  evaluación → aplicar umbrales.

In [ ]:
def fmt(x): m, h = ic95(x); return f"{m:.3f} ± {h:.3f}"
cmi = pd.DataFrame(filas_cmi)
cmi_cal = cmi[cmi['version'] == 'calibrado'].set_index('cantidad')
resumen_final = pd.Series({
    'Fusión − texto, F1 macro (pts)': fmt(100 * (fi['f1_macro'] - comp[comp['modelo'] == 'texto'].set_index('semilla')['f1_macro'])),
    'Índice de dependencia de la imagen (A)': fmt(indice),
    'Tardía por género − fusión ingenua, F1 macro (pts) (B)':
        fmt(100 * (comp[comp['modelo'] == 'tardia_por_genero'].set_index('semilla')['f1_macro'] - fi['f1_macro'])),
    'Î(X_img; Y | X_txt), nats/película, calibrado (C)':
        f"{cmi_cal.loc['I(X_img; Y | X_txt)', 'nats']:.3f} [{cmi_cal.loc['I(X_img; Y | X_txt)', 'ic_inf']:.3f}, {cmi_cal.loc['I(X_img; Y | X_txt)', 'ic_sup']:.3f}]",
    'Temperatura imagen (D)': fmt(cal[cal['modelo'] == 'imagen']['T']),
    'Temperatura texto (D)': fmt(cal[cal['modelo'] == 'texto']['T']),
    'Temperatura fusión (D)': fmt(cal[cal['modelo'] == 'fusion_ingenua']['T']),
}, name='valor (media ± IC 95%)')
print(resumen_final.to_string())

PRE = f'{RES}/exp2_{ETIQUETAS}'
abl_genero.to_csv(f'{PRE}_ablacion_por_genero.csv')
comp.to_csv(f'{PRE}_fusion_tardia_por_semilla.csv', index=False)
cmi.to_csv(f'{PRE}_cmi.csv', index=False)
cmi_genero.to_csv(f'{PRE}_cmi_por_genero.csv')
cal.to_csv(f'{PRE}_calibracion_por_semilla.csv', index=False)
por_epoca.to_csv(f'{PRE}_por_epoca.csv', index=False)
np.savez_compressed(f'{PRE}_logits.npz',
                    **{f'{p}_{n}_s{s}': logits[(n, s)][p] for (n, s) in logits for p in ['dev', 'test']},
                    idx_dev=idx['dev'], idx_test=idx['test'], generos=np.array(GENEROS_EVAL))

torch.save({
    'etiquetas': ETIQUETAS,
    'config': {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in CONFIG.items()},
    'modalidades': MODELOS,
    'generos_entrenamiento': GENEROS,   # salida de los modelos (23 logits)
    'generos_evaluacion': GENEROS_EVAL, # columnas a las que se refieren umbrales y temperaturas
    'stats_estandarizacion': {m: (mu, sd) for m, (mu, sd) in stats_estandarizacion.items()},
    'modelos': {f'{n}_s{s}': {k: v.cpu() for k, v in m.state_dict().items()} for (n, s), m in modelos.items()},
    'umbrales': {f'{n}_s{s}': u for (n, s), u in umbrales.items()},
    'temperaturas': {f"{r['modelo']}_s{r['semilla']}": r['T'] for _, r in cal.iterrows()},
}, f'{PRE}_artefactos.pt')
print("\nGuardado en", RES, ":", sorted(f for f in os.listdir(RES) if f.startswith(f'exp2_{ETIQUETAS}')))